<a href="https://colab.research.google.com/github/davidenkoiuliia/Davidenko-Iuliia/blob/main/%D0%A2%D0%BE%D0%BF%D0%BE%D0%BB%D0%BE%D0%B3%D0%B8%D1%8F%D0%9B%D0%B8%D0%B3%D0%B0%D0%BD%D0%B4%D0%BE%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Считаем заряды для ATB***

In [ ]:
pip install rdkit

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

def compute_net_charge(pdb_file):
    mol = Chem.MolFromPDBFile(pdb_file, removeHs=False)

    charge = 0
    for atom in mol.GetAtoms():
        charge += atom.GetFormalCharge()

    return charge

net_charge = compute_net_charge(pdb_file)
print("Net Charge =", net_charge)

***Готовим ЛИГАНД***

Исправляем структуру TOP файла

In [ ]:
import re

def read_pdb(path):
    with open(path, "r") as f:
        lines = f.readlines()

    atoms = []
    other_lines = []
    for line in lines:
        if line.startswith(("ATOM", "HETATM")):
            atoms.append(line)
        else:
            other_lines.append(line)
    return atoms, other_lines


def write_pdb(path, atoms, other_lines):

    with open(path, "w") as f:
        for line in atoms:
            f.write(line)
        for line in other_lines:
            f.write(line)


def read_top(path):
    with open(path, "r") as f:
        return f.readlines()


def write_top(path, lines):
    with open(path, "w") as f:
        for line in lines:
            f.write(line)


def extract_resnrs_from_top(top_lines):

    inside_atoms = False
    resnrs = []

    for line in top_lines:
        if line.strip().startswith("[ atoms ]"):
            inside_atoms = True
            continue
        if inside_atoms:
            if line.strip().startswith("["):
                break
            if line.strip() == "" or line.strip().startswith(";"):
                continue

            cols = line.split()
            if len(cols) >= 3:
                resnrs.append(cols[2])

    return resnrs


def fix_resnr_in_pdb(pdb_atoms, resnrs_top):

    fixed = []
    n = min(len(pdb_atoms), len(resnrs_top))

    for i in range(n):
        line = list(pdb_atoms[i])
        new_resnr = str(resnrs_top[i]).rjust(4)
        line[22:26] = new_resnr
        fixed.append("".join(line))

    for i in range(n, len(pdb_atoms)):
        fixed.append(pdb_atoms[i])

    return fixed


def get_moleculetype_name(top_lines):
    for i, line in enumerate(top_lines):
        if line.strip().startswith("[ moleculetype ]")
            j = i + 1
            while j < len(top_lines):
                if top_lines[j].strip() and not top_lines[j].strip().startswith(";"):
                    return top_lines[j].split()[0], j
                j += 1
    return None, None


def fix_moleculetype_name(top_lines):
    old_name, idx = get_moleculetype_name(top_lines)
    if old_name is None:
        return top_lines

    if len(old_name) <= 3:
        return top_lines

    new_name = old_name[:3]

    new_lines = []
    for line in top_lines:
        new_lines.append(line.replace(old_name, new_name))

    return new_lines


def get_atoms_block_indices(top_lines):
    start, end = None, None
    for i, line in enumerate(top_lines):
        if line.strip().startswith("[ atoms ]"):
            start = i
            continue
        if start and line.strip().startswith("[") and not line.strip().startswith("[ atoms ]"):
            end = i
            break
    if start and not end:
        end = len(top_lines)
    return start, end


def remove_extra_hydrogens(top_lines, n_pdb_atoms):
    start, end = get_atoms_block_indices(top_lines)
    atom_lines = top_lines[start + 1 : end]

    # чистые атомные строки
    atom_data = [ln for ln in atom_lines if ln.strip() and not ln.strip().startswith(";")]

    n_top_atoms = len(atom_data)
    if n_top_atoms <= n_pdb_atoms:
        return top_lines

    n_delete = n_top_atoms - n_pdb_atoms

    # ищем N последних водородов с конца
    hydrogens_indices = []
    for i in reversed(range(len(atom_lines))):
        line = atom_lines[i].strip()
        if not line or line.startswith(";"):
            continue
        cols = line.split()
        atom_name = cols[4] if len(cols) > 4 else ""
        if atom_name.upper().startswith("H"):
            hydrogens_indices.append(i)
            if len(hydrogens_indices) == n_delete:
                break

    if len(hydrogens_indices) < n_delete:
        print("Нет водородов для удаления")
        return top_lines


    hydrogens_indices = set(hydrogens_indices)

    new_atom_block = [line for i, line in enumerate(atom_lines) if i not in hydrogens_indices]


    new_top = []
    new_top.extend(top_lines[: start + 1])
    new_top.extend(new_atom_block)
    new_top.extend(top_lines[end:])
    return new_top

def fix_files(pdb_path, top_path, out_pdb, out_top):
    pdb_atoms, pdb_other = read_pdb(pdb_path)
    top_lines = read_top(top_path)
    resnrs_top = extract_resnrs_from_top(top_lines)
    pdb_atoms_fixed = fix_resnr_in_pdb(pdb_atoms, resnrs_top)
    top_fixed = fix_moleculetype_name(top_lines)
    n_pdb_atoms = len(pdb_atoms)
    top_fixed2 = remove_extra_hydrogens(top_fixed, n_pdb_atoms)
    write_pdb(out_pdb, pdb_atoms_fixed, pdb_other)
    write_top(out_top, top_fixed2)

    print("Готово")

pdb_file = "lig4ydfATB.pdb"
top_file = "fixed.top"
out_pdb = "lig4ydfATB_FINAL.pdb"
out_top = "lig4ydfATB_FINAL.top"

fix_files(pdb_file, top_file, out_pdb, out_top)


***Готовим РЕЦЕПТОР***

In [ ]:
# Установка PDBFixer из репозитория GitHub
!pip install git+https://github.com/openmm/pdbfixer.git
!pip install openmm

In [ ]:
from google.colab import files

uploaded = files.upload()  # Выберите BRAF.pdb на вашем ПК
filename = list(uploaded.keys())[0]
print("Файл загружен:", filename)

Если есть пропущенные атомы

In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

fixer = PDBFixer(filename=filename)

fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(pH=7.0)

fixed_filename = "3sm1_fixed.pdb"
PDBFile.writeFile(fixer.topology, fixer.positions, open(fixed_filename, 'w'))
print("Исправленный PDB сохранён как", fixed_filename)